# Pipeline de Modelado: Segunda Iteración (Second Attempt)

### 🎯 Objetivos de esta Versión:
1. **Limpieza de Ruido:** Eliminación de todos los flags binarios manuales ("mayor que / menor que"), conservando únicamente ratios continuos.
2. **Transformación de Riesgo (Smoothed Target Encoding OOF):** Suavizado de la variable `banda_riesgo` y `ocupacion_region` para evitar que los árboles apliquen cortes rígidos y excluyan a clientes de riesgo medio/alto que sí convierten.
3. **Ratios de Rescate para Clientes Incomprendidos:** Incorporación de métricas de interés digital latente (`visitas_web / (numero_productos + 1)`, `saldo_promedio / (dias_inactividad + 1)`, etc.).
4. **Diversidad Arquitectónica en el Ensamblaje:** Inclusión de un **Modelo Lineal Regularizado (Ridge / Regresión Logística con StandardScaler)** junto a **LightGBM** y **CatBoost**, ensamblados mediante **Meta-Aprendizaje (Stacking / Blending ponderado por AUC)**.

In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import optuna
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
print("✅ Librerías importadas correctamente.")

## Fase 1: Carga de Datos y Tratamiento de Outliers
- Separamos `id_cliente` y `mes`.
- Aplicamos capping de outliers en percentiles extremos ($0.1\%$ y $99.5\%$) y transformaciones $\log(1+x)$.

In [ ]:
# Cargar datos
train_df = pd.read_csv('../../dataset/train.csv') if os.path.exists('../../dataset/train.csv') else pd.read_csv('dataset/train.csv')
test_df = pd.read_csv('../../dataset/test.csv') if os.path.exists('../../dataset/test.csv') else pd.read_csv('dataset/test.csv')

test_ids = test_df['id_cliente'].copy()
y = train_df['objetivo'].copy()

cols_to_drop = ['id_cliente', 'mes']
X_train = train_df.drop(columns=cols_to_drop + ['objetivo']).copy()
X_test = test_df.drop(columns=cols_to_drop).copy()

# Capping de outliers numéricos
outlier_cols = ['ingresos', 'saldo_promedio', 'ratio_deuda_ingresos', 'distancia_sucursal_km']
for col in outlier_cols:
    upper_limit = X_train[col].quantile(0.995)
    lower_limit = X_train[col].quantile(0.001)
    X_train[col] = X_train[col].clip(lower=lower_limit, upper=upper_limit)
    X_test[col] = X_test[col].clip(lower=lower_limit, upper=upper_limit)
    
    X_train[f'log_{col}'] = np.log1p(np.maximum(0, X_train[col]))
    X_test[f'log_{col}'] = np.log1p(np.maximum(0, X_test[col]))

print(f"Train base: {X_train.shape} | Test base: {X_test.shape}")

## Fase 2: Feature Engineering Limpio & Ratios de Rescate
- **Eliminación de flags binarios ruidosos** (sin comparaciones artificiales).
- **Ratios de rescate:** `visitas_web / (numero_productos + 1)`, `saldo_promedio / (inactividad + 1)`, `potencial_digital_latente`.
- **Estadísticas grupales continuas:** Ratios de saldo e ingresos respecto a la media de `region`, `ocupacion` y `canal`.

In [ ]:
def generate_clean_features(train, test):
    tr = train.copy()
    te = test.copy()
    
    for d in [tr, te]:
        # 1. Ratios Financieros Básicos
        d['capacidad_ahorro'] = d['saldo_promedio'] / (d['ingresos'] + 1.0)
        d['intensidad_productos'] = d['saldo_promedio'] / (d['numero_productos'] + 0.001)
        d['dias_inactividad_relativa'] = d['dias_ultima_transaccion'] - d['dias_ultima_interaccion']
        d['deuda_total_estimada'] = d['ingresos'] * d['ratio_deuda_ingresos']
        d['proporcion_deuda_saldo'] = d['deuda_total_estimada'] / (d['saldo_promedio'] + 1.0)
        d['saldo_neto_estimado'] = d['saldo_promedio'] - d['deuda_total_estimada']
        d['saldo_por_edad'] = d['saldo_promedio'] / (d['edad'] + 1.0)
        d['ingresos_por_edad'] = d['ingresos'] / (d['edad'] + 1.0)
        
        # 2. Ratios de Rescate (Digital vs Vinculación Tradicional)
        inactividad_no_negativa = np.maximum(0, d['dias_inactividad_relativa'])
        d['ratio_visitas_productos'] = d['visitas_web_ultimos_90_dias'] / (d['numero_productos'] + 1.0)
        d['ratio_saldo_inactividad'] = d['saldo_promedio'] / (inactividad_no_negativa + 1.0)
        d['ratio_visitas_inactividad'] = d['visitas_web_ultimos_90_dias'] / (inactividad_no_negativa + 1.0)
        d['potencial_digital_latente'] = (d['visitas_web_ultimos_90_dias'] + d['activo_movil'].astype(int) * 3.0) / (d['numero_productos'] + 1.0)
        d['ratio_saldo_transaccion'] = d['saldo_promedio'] / (d['dias_ultima_transaccion'] + 1.0)
        
        # 3. Cruces Categóricos Clave
        d['ocupacion_region'] = d['ocupacion'].astype(str) + '_' + d['region'].astype(str)
        d['canal_banda'] = d['canal_adquisicion'].astype(str) + '_' + d['banda_riesgo'].astype(str)

    # 4. Estadísticas Agrupadas Continuas (Sin Flags Booleanos)
    # Por Región
    grp_region = tr.groupby('region')[['saldo_promedio', 'ingresos']].agg(['mean', 'std']).reset_index()
    grp_region.columns = ['region', 'grp_reg_saldo_mean', 'grp_reg_saldo_std', 'grp_reg_ing_mean', 'grp_reg_ing_std']
    tr = tr.merge(grp_region, on='region', how='left')
    te = te.merge(grp_region, on='region', how='left')

    # Por Ocupación
    grp_ocup = tr.groupby('ocupacion')[['saldo_promedio', 'ingresos']].agg(['mean', 'std']).reset_index()
    grp_ocup.columns = ['ocupacion', 'grp_ocup_saldo_mean', 'grp_ocup_saldo_std', 'grp_ocup_ing_mean', 'grp_ocup_ing_std']
    tr = tr.merge(grp_ocup, on='ocupacion', how='left')
    te = te.merge(grp_ocup, on='ocupacion', how='left')

    # Ratios relativos contra la media del grupo
    for d in [tr, te]:
        d['ratio_saldo_vs_region'] = d['saldo_promedio'] / (d['grp_reg_saldo_mean'] + 1.0)
        d['ratio_ingresos_vs_region'] = d['ingresos'] / (d['grp_reg_ing_mean'] + 1.0)
        d['ratio_saldo_vs_ocup'] = d['saldo_promedio'] / (d['grp_ocup_saldo_mean'] + 1.0)
        d['ratio_ingresos_vs_ocup'] = d['ingresos'] / (d['grp_ocup_ing_mean'] + 1.0)

    return tr, te

X_train, X_test = generate_clean_features(X_train, X_test)
print(f"✅ Features continuas generadas. Total columnas: {X_train.shape[1]}")

## Fase 3: Target Encoding Suavizado (Smoothed Target Encoding OOF)
Para evitar que `banda_riesgo` actúe como un corte binario excluyente, la convertimos en su probabilidad continua suavizada de conversión:
$$TE(c) = \frac{n_c \cdot \bar{y}_c + m \cdot \bar{y}_{global}}{n_c + m}$$
Se calcula Out-Of-Fold (OOF) en los 5 pliegues de entrenamiento para evitar fuga de datos (Data Leakage).

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def apply_smoothed_target_encoding(X_tr, y_tr, X_te, cols, smoothing=20.0):
    global_mean = y_tr.mean()
    X_tr_encoded = X_tr.copy()
    X_te_encoded = X_te.copy()
    
    for col in cols:
        # 1. Encoding OOF para entrenamiento
        oof_te = np.zeros(len(X_tr))
        for train_idx, val_idx in skf.split(X_tr, y_tr):
            fold_train = X_tr.iloc[train_idx]
            fold_y = y_tr.iloc[train_idx]
            fold_val = X_tr.iloc[val_idx]
            
            stats = fold_train.groupby(col).apply(lambda g: pd.Series({
                'count': len(g),
                'mean': fold_y.iloc[g.index].mean()
            }), include_groups=False)
            
            smooth_dict = {}
            for cat, row in stats.iterrows():
                smooth_dict[cat] = (row['count'] * row['mean'] + smoothing * global_mean) / (row['count'] + smoothing)
                
            oof_te[val_idx] = fold_val[col].map(smooth_dict).fillna(global_mean)
            
        X_tr_encoded[f'{col}_te'] = oof_te
        
        # 2. Encoding completo para test
        full_stats = X_tr.groupby(col).apply(lambda g: pd.Series({
            'count': len(g),
            'mean': y_tr.iloc[g.index].mean()
        }), include_groups=False)
        
        full_dict = {}
        for cat, row in full_stats.iterrows():
            full_dict[cat] = (row['count'] * row['mean'] + smoothing * global_mean) / (row['count'] + smoothing)
            
        X_te_encoded[f'{col}_te'] = X_te[col].map(full_dict).fillna(global_mean)
        
    return X_tr_encoded, X_te_encoded

X_train, X_test = apply_smoothed_target_encoding(X_train, y, X_test, cols=['banda_riesgo', 'ocupacion_region'], smoothing=25.0)
print("✅ Smoothed Target Encoding OOF aplicado con éxito a 'banda_riesgo' y 'ocupacion_region'.")

## Fase 4: Optimización Optuna Leaf-Wise para LightGBM
- `max_depth = -1` (crecimiento asimétrico).
- `num_leaves` en `[31, 255]`.
- Regularización L1 (`reg_alpha`) y L2 (`reg_lambda`).

In [ ]:
cat_cols = ['ocupacion', 'region', 'canal_adquisicion', 'banda_riesgo', 'dispositivo_principal',
            'ocupacion_region', 'canal_banda']

for col in cat_cols:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')

def objective(trial):
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'max_depth': -1,
        'num_leaves': trial.suggest_int('num_leaves', 31, 255),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.12, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 150),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'subsample_freq': 1,
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 0.95),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-6, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-6, 10.0, log=True),
        'n_estimators': 2000,
        'random_state': 42,
        'verbose': -1,
        'n_jobs': -1
    }
    
    auc_scores = []
    for train_idx, val_idx in skf.split(X_train, y):
        X_tr, y_tr = X_train.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X_train.iloc[val_idx], y.iloc[val_idx]
        
        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
        )
        
        preds = model.predict_proba(X_va)[:, 1]
        auc_scores.append(roc_auc_score(y_va, preds))
        
    return np.mean(auc_scores)

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50, show_progress_bar=True)

best_lgb_params = study.best_params
print("Mejores Hiperparámetros LightGBM:", best_lgb_params)
print(f"Mejor AUC CV promedio: {study.best_value:.5f}")

## Fase 5: Entrenamiento Multimodelo con Diversidad Lineal
Entrenamos 3 modelos con naturalezas matemáticas diferentes:
1. **LightGBM:** Árboles con optimización leaf-wise.
2. **CatBoost:** Árboles simétricos (oblivious trees) con manejo nativo de categorías.
3. **Regresión Logística Regularizada (Ridge / L2):** Función sigmoide suave y continua con escalado estándar `StandardScaler`.

In [ ]:
# Arrays para predicciones OOF y Test
oof_lgb = np.zeros(len(X_train))
test_preds_lgb = np.zeros(len(X_test))

oof_cb = np.zeros(len(X_train))
test_preds_cb = np.zeros(len(X_test))

oof_lr = np.zeros(len(X_train))
test_preds_lr = np.zeros(len(X_test))

# -------------------------------------------------------------------------
# 1. Entrenamiento LightGBM
# -------------------------------------------------------------------------
print("1️⃣ Entrenando LightGBM...")
final_lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'max_depth': -1,
    'n_estimators': 3000,
    'subsample_freq': 1,
    'random_state': 42,
    'verbose': -1,
    'n_jobs': -1,
    **best_lgb_params
}

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y)):
    X_tr, y_tr = X_train.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X_train.iloc[val_idx], y.iloc[val_idx]

    model_lgb = lgb.LGBMClassifier(**final_lgb_params)
    model_lgb.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    oof_lgb[val_idx] = model_lgb.predict_proba(X_va)[:, 1]
    test_preds_lgb += model_lgb.predict_proba(X_test)[:, 1] / skf.n_splits

print(f"   👉 AUC LightGBM OOF: {roc_auc_score(y, oof_lgb):.5f}")

# -------------------------------------------------------------------------
# 2. Entrenamiento CatBoost
# -------------------------------------------------------------------------
print("\n2️⃣ Entrenando CatBoost...")
X_train_cb = X_train.copy()
X_test_cb = X_test.copy()
for col in cat_cols:
    X_train_cb[col] = X_train_cb[col].astype(str)
    X_test_cb[col] = X_test_cb[col].astype(str)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_cb, y)):
    X_tr, y_tr = X_train_cb.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X_train_cb.iloc[val_idx], y.iloc[val_idx]

    model_cb = CatBoostClassifier(
        eval_metric='AUC',
        random_seed=42,
        early_stopping_rounds=50,
        verbose=False,
        thread_count=-1
    )

    train_pool = Pool(X_tr, y_tr, cat_features=cat_cols)
    val_pool = Pool(X_va, y_va, cat_features=cat_cols)
    test_pool = Pool(X_test_cb, cat_features=cat_cols)

    model_cb.fit(train_pool, eval_set=val_pool, use_best_model=True)
    oof_cb[val_idx] = model_cb.predict_proba(val_pool)[:, 1]
    test_preds_cb += model_cb.predict_proba(test_pool)[:, 1] / skf.n_splits

print(f"   👉 AUC CatBoost OOF: {roc_auc_score(y, oof_cb):.5f}")

# -------------------------------------------------------------------------
# 3. Entrenamiento Regresión Logística (Diversidad Lineal / Suave)
# -------------------------------------------------------------------------
print("\n3️⃣ Entrenando Modelo Lineal Regularizado (Ridge / Regresión Logística)...")
num_cols = [c for c in X_train.columns if c not in cat_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y)):
    X_tr, y_tr = X_train.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X_train.iloc[val_idx], y.iloc[val_idx]

    X_tr_proc = preprocessor.fit_transform(X_tr)
    X_va_proc = preprocessor.transform(X_va)
    X_te_proc = preprocessor.transform(X_test)

    model_lr = LogisticRegression(C=0.1, penalty='l2', max_iter=1000, random_state=42)
    model_lr.fit(X_tr_proc, y_tr)

    oof_lr[val_idx] = model_lr.predict_proba(X_va_proc)[:, 1]
    test_preds_lr += model_lr.predict_proba(X_te_proc)[:, 1] / skf.n_splits

print(f"   👉 AUC Modelo Lineal OOF: {roc_auc_score(y, oof_lr):.5f}")

## Fase 6: Ensamblaje Óptimo por Meta-Aprendizaje (Blending Stacking)
En lugar del promedio simple, optimizamos directamente los pesos $[w_1, w_2, w_3]$ de cada arquitectura para maximizar el AUC global:

In [ ]:
# Matriz OOF de los 3 modelos
OOF_matrix = np.column_stack([oof_lgb, oof_cb, oof_lr])
TEST_matrix = np.column_stack([test_preds_lgb, test_preds_cb, test_preds_lr])

# Optimización de pesos para maximizar AUC
def loss_function(weights):
    weights = np.array(weights)
    weights = weights / np.sum(weights)
    blend_preds = np.dot(OOF_matrix, weights)
    return -roc_auc_score(y, blend_preds)

initial_weights = [1/3, 1/3, 1/3]
bounds = [(0, 1), (0, 1), (0, 1)]
constraints = ({'type': 'eq', 'fun': lambda w: 1 - sum(w)})

res = minimize(loss_function, initial_weights, method='SLSQP', bounds=bounds, constraints=constraints)
best_weights = res.x / np.sum(res.x)

print("=" * 65)
print("⚖️ PESOS ÓPTIMOS DEL ENSAMBLAJE MULTIARQUITECTURA:")
print(f"   - Peso LightGBM (Leaf-Wise):          {best_weights[0]*100:.1f}%")
print(f"   - Peso CatBoost (Oblivious Trees):    {best_weights[1]*100:.1f}%")
print(f"   - Peso Modelo Lineal (Ridge/Suave):   {best_weights[2]*100:.1f}%")
print("=" * 65)

# Predicciones finales ensambladas
oof_final = np.dot(OOF_matrix, best_weights)
final_test_predictions = np.dot(TEST_matrix, best_weights)

final_auc = roc_auc_score(y, oof_final)
final_gini = 2.0 * final_auc - 1.0

print(f"\n⭐ AUC ESTIMADO FINAL (OOF):  {final_auc:.5f}")
print(f"🏆 GINI ESTIMADO FINAL:        {final_gini:.5f}")
print("=" * 65)

# Guardar archivo de submission final
submission_df = pd.DataFrame({
    'id_cliente': test_ids,
    'prediccion': final_test_predictions
})

sub_path = 'submission_second_attempt.csv'
submission_df.to_csv(sub_path, index=False)
print(f"\n✅ Archivo '{sub_path}' generado con éxito.")
submission_df.head()